In [ ]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

import numpy as np
import matplotlib.pyplot as plt
from src.generators import Math2DGenerator
from src.noise import AbsoluteGaussianNoise
from src.shadows import BandShadow
from src.models import RandomForestModel
from src.splitter import GapSplitter
from src.orchestrator import RegressionOrchestator
from src.visualization import plot_experiment_results

N_SAMPLES = 400
X_RANGE = (6, 14)
SHADOW_RANGE = (9, 11)

generator = Math2DGenerator(function=lambda x: np.where(x > 10, 15, 5), seed=42)
noise_strategy = AbsoluteGaussianNoise(std_dev=0.1, seed=42)
shadow = BandShadow(axis='x', min_val=SHADOW_RANGE[0], max_val=SHADOW_RANGE[1])
splitter = GapSplitter(test_size=0.2, random_state=42)
model = RandomForestModel(n_estimators=100, max_depth=10, random_state=42)
orchestrator = RegressionOrchestator(generator, splitter, model)

print(f"Esperimentua hasten {model.get_name()} modeloa erabiliz...")

artifacts = orchestrator.run(
    n_samples=N_SAMPLES, 
    x_range=X_RANGE, 
    noise_strategy=noise_strategy, 
    shadow=shadow
)

mse_vis = artifacts.metrics.mse_test_visible
mse_sha = artifacts.metrics.mse_test_shadow

print(f"\n--- Errore kalkuluak (MSE) ---")
print(f"Ikusgarria: {mse_vis:.5f}")
print(f"Itzaldutakoa: {mse_sha:.5f}")

if artifacts.uncertainties:
    std_vis = artifacts.uncertainties['visible']
    std_sha = artifacts.uncertainties['shadow']
    
    mean_std_vis = np.mean(std_vis)
    mean_std_sha = np.mean(std_sha)
    
    print(f"\n--- Balioetsitako ezjakintaasuna (BB bariantza) ---")
    print(f"Eremu ikusgarria: {mean_std_vis:.5f} (Konfidantza altua)")
    print(f"Itzaldutako eremua: {mean_std_sha:.5f} (Konfidantza baxua / Ezjakintasun altua)")
    
    if mean_std_sha > mean_std_vis:
        print("\n[SUCCESS] Ereduak ziurgabetasun handiagoa ematen du eremu ezezagunean.")
    else:
        print("\n[WARNING] Eredua ez da ondo detektatzen ari itzaleko ziurgabetasuna.")
else:
    print("\n[ERROR] Ez da ziurgabetasun-daturik sortu.")

plt.figure(figsize=(12, 8))
plot_experiment_results(
    artifacts.split_data, 
    artifacts.predictions, 
    uncertainties=artifacts.uncertainties, 
    title=f"Exp 2.0.1: Ezjakintasuna {model.get_name()} modeloarekin\nItzala: {SHADOW_RANGE}"
)